In [1]:
# --- Requirements ---
# pip install rasterio numpy

import rasterio
import numpy as np

# --- Input raster paths ---
tif_files = [
    r"C:\Users\dpn_s\Documents\PhD_Research\Salinity_Model_Matagorda\Simulation_results\2024\Jan-Feb24_stat.tif",
    r"C:\Users\dpn_s\Documents\PhD_Research\Salinity_Model_Matagorda\Simulation_results\2024\Mar-Apr24_stat.tif",
    r"C:\Users\dpn_s\Documents\PhD_Research\Salinity_Model_Matagorda\Simulation_results\2024\May-Jun24_stat.tif",
    r"C:\Users\dpn_s\Documents\PhD_Research\Salinity_Model_Matagorda\Simulation_results\2024\Jul-Aug24_stat.tif",
    r"C:\Users\dpn_s\Documents\PhD_Research\Salinity_Model_Matagorda\Simulation_results\2024\Sep-Oct24_stat.tif",
    r"C:\Users\dpn_s\Documents\PhD_Research\Salinity_Model_Matagorda\Simulation_results\2024\Nov-Dec24_stat.tif"
]

# --- Output raster path ---
out_tif = r"C:\Users\dpn_s\Documents\PhD_Research\Salinity_Model_Matagorda\Simulation_results\2024\Mean_2024.tif"

# --- Read all rasters and compute cell-wise mean ---
arrays = []
ref_meta = None

for path in tif_files:
    with rasterio.open(path) as src:
        data = src.read(1).astype(float)
        nodata = src.nodata
        if ref_meta is None:
            ref_meta = src.meta.copy()
        # Replace nodata with NaN for proper averaging
        if nodata is not None:
            data = np.where(data == nodata, np.nan, data)
        arrays.append(data)

# Stack into 3D array (n_files, rows, cols)
stack = np.stack(arrays, axis=0)

# Compute mean ignoring NaN
mean_array = np.nanmean(stack, axis=0)

# Replace NaN with nodata again
if ref_meta.get("nodata") is not None:
    mean_array = np.where(np.isnan(mean_array), ref_meta["nodata"], mean_array)

# --- Write output raster ---
ref_meta.update(dtype=rasterio.float32, count=1)
with rasterio.open(out_tif, "w", **ref_meta) as dst:
    dst.write(mean_array.astype(np.float32), 1)

print(f"✅ Mean raster saved to: {out_tif}")


✅ Mean raster saved to: C:\Users\dpn_s\Documents\PhD_Research\Salinity_Model_Matagorda\Simulation_results\2024\Mean_2024.tif


C:\Users\dpn_s\AppData\Local\Temp\ipykernel_38996\2551669986.py:39: RuntimeWarning: Mean of empty slice
  mean_array = np.nanmean(stack, axis=0)
